# Data Science Project - Phase 3: Features and Target
**The Context Gap**
Mohammad Qasim Khalaf Alsamarrae

In this notebook I build my own features from the clean table of Phase 2, and the target: **did the blind person get the context they needed?**

Every row answers four questions:
1. What did the person need? (question type)
2. What was the photo like? (visibility and framing problems)
3. What was in the photo? (descriptions: did they cover the need, did they mention the place)
4. What did the person get? (the context gap and its source)

In [ ]:
# Import Pandas and numpy
import numpy as np
import pandas as pd
import re   # used only to find place words in the descriptions

import os

# clean_data.csv comes from the Phase 2 notebook.
# In Colab every notebook runs on its own machine, so if the file is not here,
# download it from notebook 2 (folder icon on the left) and upload it now.
if not os.path.exists("clean_data.csv"):
    from google.colab import files
    print("clean_data.csv not found: please upload it")
    files.upload()

# keep_default_na=False keeps answers like "n/a" or "null" as text instead of empty values
data = pd.read_csv("clean_data.csv", keep_default_na=False)
data.shape

## 1. Question type
I give every question one of 7 types. The order of the checks matters: "what color is this?" has "what" and "color", so color is checked first.

1. color
2. count (how many, how much)
3. read_text (read, say, title, label, date, number...)
4. scene_place (describe, look like, where, outside...)
5. identify (what, which, kind, brand, flavor...)
6. yes_no (starts with is, are, does, can...)
7. other

In [ ]:
color_words = ["color", "colour", "colors", "colours"]
count_words = ["how many", "how much"]
read_words = ["read", "say", "says", "said", "written", "text", "screen", "title", "label", "instructions",
              "directions", "number", "numbers", "code", "date", "expiration", "expire", "expires", "price",
              "ingredients", "words", "writing", "temperature", "time", "serial", "model", "page", "letter", "sign"]
scene_words = ["describe", "description", "look like", "looks like", "looking at", "where", "scene",
               "picture of", "outside", "room", "sky", "around"]
identify_words = ["what", "what's", "whats", "which", "who", "identify", "kind", "type", "brand", "flavor", "flavour"]
yes_no_starts = ["is", "are", "does", "do", "can", "could", "will", "would", "did", "was", "has", "have", "should"]

def has_any(text, word_list):
    # spaces around the text so "time" does not match inside "sometimes"
    for p in "?,.!;:":
        text = text.replace(p, " ")
    text = " " + " ".join(text.split()) + " "
    for w in word_list:
        if " " + w + " " in text:
            return True
    return False

def question_type(q):
    if has_any(q, color_words):
        return "color"
    if has_any(q, count_words):
        return "count"
    if has_any(q, read_words):
        return "read_text"
    if has_any(q, scene_words):
        return "scene_place"
    if has_any(q, identify_words):
        return "identify"
    words = q.replace("?", " ").replace(",", " ").replace(".", " ").split()
    if len(words) > 0 and words[0] in yes_no_starts:
        return "yes_no"
    return "other"

data["question_type"] = data["question_clean"].apply(question_type)
data["question_type"].value_counts()

In [ ]:
# percent
(data["question_type"].value_counts(normalize=True) * 100).round(1)

## 2. Photo problems: visibility and framing
I split the six problems into two groups, because they need different advice for the blind person:
- **visibility** (BLR blur, DRK dark, BRT too bright): the photo itself does not show things clearly → "hold steady, move to the light"
- **framing** (FRM bad framing, OBS blocked, ROT rotated): the photo is clear, but the object is cut, covered or upside down → "move the phone, remove your finger"

A problem counts if 3 or more of the 5 workers saw it.

In [ ]:
data["visibility_problems"] = (data[["BLR", "DRK", "BRT"]] >= 3).sum(axis=1)
data["framing_problems"] = (data[["FRM", "OBS", "ROT"]] >= 3).sum(axis=1)

print(data["visibility_problems"].value_counts().sort_index())
print(data["framing_problems"].value_counts().sort_index())

## 3. Photos nobody could recognize
**cant_describe = 1** if 3 or more workers said they could not recognize anything in the photo.

I checked unrecognizable against precanned_count from Phase 2: they are almost the same (correlation about 0.97), so they come from the same task. I use only unrecognizable here.

In [ ]:
print(data[["unrecognizable", "precanned_count"]].corr().round(2))
data["cant_describe"] = (data["unrecognizable"] >= 3).astype(int)
data["cant_describe"].value_counts()

## 4. Did the descriptions cover the need?
The describers did not see the question. I check if the descriptions still contain what the person needed:
- **read_text**: do the descriptions give the text itself? (numbers, quotes, or words like "says", "reads", "titled")
- **color**: do they name a color?
- **other types**: do all the words of the top answer appear in the descriptions?

Result: **covers**, **misses**, **no_description** (no valid description) or **unknown** (the answer itself is "unanswerable", "yes" or "no", so I can't know what information was needed).

In [ ]:
colors = ["red", "blue", "green", "yellow", "white", "black", "gray", "grey", "brown", "pink", "purple",
          "orange", "beige", "tan", "silver", "gold", "navy", "maroon", "teal", "cream"]
text_signs = ["says", "say", "reads", "reading", "labeled", "labelled", "titled", "written", "brand", "logo"]

def gives_text(caption):
    if '"' in caption or "“" in caption or "”" in caption:
        return True
    for ch in caption:
        if ch.isdigit():
            return True
    return has_any(caption.lower(), text_signs)

def coverage(row):
    caption = row["caption"].lower()
    top = str(row["top_answer"])
    if row["valid_captions"] == 0:
        return "no_description"
    if row["question_type"] == "read_text":
        return "covers" if gives_text(row["caption"]) else "misses"
    if row["question_type"] == "color":
        return "covers" if has_any(caption, colors) else "misses"
    if top in ["unanswerable", "yes", "no", ""]:
        return "unknown"
    words = [w for w in top.split() if len(w) > 2]
    if len(words) > 0 and all(w in caption for w in words):
        return "covers"
    return "misses"

data["description_coverage"] = data.apply(coverage, axis=1)
data["description_coverage"].value_counts()

In [ ]:
# by question type
pd.crosstab(data["question_type"], data["description_coverage"])

## 5. Did the descriptions mention the place?
**place_count** = how many of the valid descriptions say where the object is: a place word (on, in, next to, under...) followed by a place (table, floor, kitchen, hand...).

In [ ]:
places = ["table", "desk", "counter", "countertop", "floor", "bed", "couch", "sofa", "chair", "shelf", "wall",
          "kitchen", "room", "bathroom", "bedroom", "office", "carpet", "rug", "tile", "ground", "street", "road",
          "outside", "outdoors", "window", "door", "car", "hand", "lap", "sink", "stove", "refrigerator", "fridge",
          "cabinet", "drawer", "yard", "grass", "sky", "store", "shop", "background", "surface", "tray", "plate",
          "bench", "porch", "building", "house", "garden", "park", "sidewalk", "dresser", "nightstand", "mat",
          "blanket", "towel", "palm", "fingers"]
place_pattern = re.compile(r"\b(on|in|next to|beside|near|against|under|inside|atop|behind|in front of)\b.*?\b("
                           + "|".join(places) + r")s?\b", re.IGNORECASE)

def place_count(caption):
    if caption == "":
        return 0
    return sum(1 for c in caption.split(" | ") if place_pattern.search(c))

data["place_count"] = data["caption"].apply(place_count)
data["place_count"].value_counts().sort_index()

In [ ]:
# photos where at least one description mentions the place (only photos with descriptions)
with_desc = data[data["valid_captions"] > 0]
(with_desc["place_count"] > 0).mean() * 100

## 6. Answer kind
I give the top answer a kind, so the information is kept and can be analysed later:
**unanswerable**, **yes_no**, **number**, **color**, **container** (only names the container, like "can" or "box", which the person already knows by touch) or **specific**.

In [ ]:
container_answers = ["can", "box", "bottle", "jar", "package", "packet", "bag", "container", "paper", "food",
                     "product", "item", "object", "thing", "stuff", "label", "text", "writing", "words",
                     "letters", "nothing", "card", "tin", "carton", "cup"]
number_words = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten",
                "eleven", "twelve", "twenty", "hundred"]

def answer_kind(top):
    top = str(top)
    if top == "unanswerable":
        return "unanswerable"
    if top in ["yes", "no"]:
        return "yes_no"
    words = top.split()
    if all(w in number_words or all(ch.isdigit() or ch in "$%/:-" for ch in w) for w in words):
        return "number"
    if all(w in colors + ["light", "dark", "and"] for w in words) and any(w in colors for w in words):
        return "color"
    if top in container_answers:
        return "container"
    return "specific"

data["answer_kind"] = data["top_answer"].apply(answer_kind)
data["answer_kind"].value_counts()

## 7. The target: context gap
For every message that is a question or a request, I check the reasons in this order (each row gets only one reason):
1. **unanswerable**: nobody could answer
2. **disagreement**: the volunteers really disagreed (agreement by meaning 2 or fewer)
3. **detail_varies**: they saw the same thing but gave different levels of detail (exact agreement 2 or fewer, following the official VQA rule that 3 people make an answer reliable)
4. **generic**: the answer kind is "container", which the person already knows by touch
5. **none**: the person got a clear answer

**context_gap = 1** if the reason is not "none". Messages of type "other" (no clear need) are **excluded**, because we don't know what the person needed.

In [ ]:
def gap_source(row):
    if row["message_type"] == "other":
        return "excluded"
    if row["answerable"] == 0:
        return "unanswerable"
    if row["agreement_meaning"] <= 2:
        return "disagreement"
    if row["agreement_exact"] <= 2:
        return "detail_varies"
    if row["answer_kind"] == "container":
        return "generic"
    return "none"

data["gap_source"] = data.apply(gap_source, axis=1)
data["context_gap"] = np.where(data["gap_source"] == "excluded", np.nan,
                               np.where(data["gap_source"] == "none", 0, 1))
data["gap_source"].value_counts()

In [ ]:
# the context gap (strict definition), without the excluded rows
used = data[data["gap_source"] != "excluded"]
print("questions used:", len(used))
print("context gap:", int(used["context_gap"].sum()), "=", round(used["context_gap"].mean() * 100, 1), "%")

# lenient definition: "detail_varies" is not counted as a gap
lenient = used["gap_source"].isin(["unanswerable", "disagreement", "generic"])
print("lenient definition:", lenient.sum(), "=", round(lenient.mean() * 100, 1), "%")

In [ ]:
# context gap by question type (%)
(used.groupby("question_type")["context_gap"].mean() * 100).round(1).sort_values(ascending=False)

### Real examples of each reason

In [ ]:
for reason in ["unanswerable", "disagreement", "detail_varies", "generic", "none"]:
    print("====", reason)
    sample = data[data["gap_source"] == reason].head(3)
    for _, r in sample.iterrows():
        print("Q:", r["question"][:80])
        print("   answer:", r["top_answer"], "| exact:", r["agreement_exact"], "| meaning:", r["agreement_meaning"])
    print()

## 8. Save
- **features_full.csv**: everything
- **analysis_table.csv**: only the columns needed for the analysis in Phase 4

In [ ]:
data.to_csv("features_full.csv", index=False)

analysis_columns = ["image", "question", "message_type", "question_type", "top_answer", "answer_kind",
                    "agreement_exact", "agreement_meaning", "visibility_problems", "framing_problems",
                    "cant_describe", "text_detected", "valid_captions", "description_coverage",
                    "place_count", "caption", "gap_source", "context_gap"]
analysis = data[analysis_columns]
analysis.to_csv("analysis_table.csv", index=False)
print(data.shape, analysis.shape)

## Notes
- Question types: identify 2,232 (51.7%), read_text 1,039 (24.1%), color 378 (8.8%), yes_no 243 (5.6%), other 201 (4.7%), scene_place 174 (4.0%), count 52 (1.2%). The rules make some mistakes (for example "describe this shirt" is counted as scene_place).
- Visibility problems in 1,501 photos (34.8%), framing problems in 1,994 photos (46.2%).
- 343 photos (7.9%) could not be recognized by most workers (cant_describe).
- Descriptions and the need: for read_text questions the descriptions did not give the text in 681 of 1,039 cases (65.5%). They say there is text, but not what it says. For color questions they named a color in 347 of 378 (91.8%). 968 rows are "unknown" because the answer itself was unanswerable, yes or no.
- In 61.7% of photos with descriptions, at least one description says where the object is.
- Answer kind: specific 2,249, unanswerable 1,385, color 334, yes_no 229, number 76, container 46.
- Context gap (strict): 2,114 of 4,244 questions (49.8%). Sources: unanswerable 1,326, detail_varies 469, disagreement 289, generic 30. 75 "other" messages were excluded.
- Context gap (lenient, without detail_varies): 1,645 (38.8%). So between about 40% and half of the blind users did not get the information they needed, even from sighted people.
- By question type: count 71.2% (only 52 questions), read_text 66.7%, scene_place 65.1%, identify 45.7%, yes_no 37.9%, color 19.0%.
- Only 16 questions (0.4%) had two reasons at the same time, so the order of the checks almost never matters.
- Human volunteers almost never give a container answer like "can" (only 46 cases). Generic answers are known to be a problem of AI tools, which can be tested in a future step.

Next step: Phase 4, analysing the relationships.